<div style="color:#3c4d5a; border-top:7px solid #42A5F5; border-bottom:7px solid #42A5F5; padding:8px; text-align:center; text-transform:uppercase">
  <h1>03. PREDICCIÓN Y PRUEBAS DE INFERENCIA</h1>
</div>

Este notebook demuestra cómo cargar la versión congelada de los modelos (Phase 1 Final Release) y realizar inferencia sobre nuevos ejemplos de manera segura, aislando completamente el proceso de entrenamiento.


In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import hashlib


## 1. Verificación de Seguridad y Manifiesto
Antes de cargar los pesos del modelo, verificamos el archivo `manifest.json` y la integridad criptográfica de los archivos para garantizar que no han sido alterados (Data Poisoning).

In [ ]:
RELEASE_DIR = 'models/phase1_final_release'
MANIFEST_PATH = os.path.join(RELEASE_DIR, 'manifest.json')

with open(MANIFEST_PATH, 'r') as f:
    manifest = json.load(f)

print(f"Versión de Liberación: {manifest['release_name']} ({manifest['release_version']})")

def sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as file:
        while chunk := file.read(8192):
            h.update(chunk)
    return h.hexdigest()

print("\nVerificando hashes...")
for filename, expected_hash in manifest['sha256_hashes'].items():
    filepath = os.path.join(RELEASE_DIR, filename)
    actual_hash = sha256(filepath)
    if actual_hash == expected_hash:
        print(f"[OK] {filename}")
    else:
        print(f"[ERROR] {filename} HASH MISMATCH!")


## 2. Carga de Artefactos de ML
Cargamos los preprocesadores y los clasificadores desde el directorio de liberación seguro.

In [ ]:
# Nombres definidos en el manifiesto
mod_r = joblib.load(os.path.join(RELEASE_DIR, manifest['reactive_model_path']))
mod_p = joblib.load(os.path.join(RELEASE_DIR, manifest['predictive_model_path']))
prep_r = joblib.load(os.path.join(RELEASE_DIR, manifest['reactive_preprocessor_path']))
prep_p = joblib.load(os.path.join(RELEASE_DIR, manifest['predictive_preprocessor_path']))

with open(os.path.join(RELEASE_DIR, manifest['metadata_path']), 'r') as f:
    meta = json.load(f)

print("Modelos cargados correctamente en memoria.")


## 3. Prueba de Inferencia: Modelo Reactivo
Tomamos una muestra aleatoria de datos, aplicamos la estandarización y realizamos la predicción.

In [ ]:
df_r = pd.read_csv('data/processed/dataset_reactivo.csv')
if 'ping_ms' in df_r.columns:
    df_r.rename(columns={'ping_ms': 'latency_ms'}, inplace=True)

sample_r = df_r[df_r['split'] == 'test'].sample(5, random_state=42)
X_r = sample_r[meta['reactive_features']]
y_true_r = sample_r['recommended_profile']

# Pipeline de inferencia
X_r_scaled = prep_r.transform(X_r)
preds_r = mod_r.predict(X_r_scaled)

print("--- INFERENCIA REACTIVA ---")
for i, (idx, row) in enumerate(X_r.iterrows()):
    print(f"Muestra {i+1} | Input: Upload={row['upload_mbps']:.2f}, Download={row['download_mbps']:.2f}, Latency={row['latency_ms']:.2f}")
    print(f"           | Predicción: {preds_r[i]} (Real: {y_true_r.iloc[i]})\n")


## 4. Prueba de Inferencia: Modelo Predictivo
Tomamos una muestra de las ventanas temporales, aplicamos el preprocesador, generamos el vector de probabilidades y aplicamos el **umbral customizado (0.55)** para la decisión final.

In [ ]:
df_p = pd.read_csv('data/processed/dataset_predictivo.csv')
sample_p = df_p[df_p['split'] == 'test'].sample(5, random_state=42)
X_p = sample_p[meta['predictive_features']]
y_true_p = sample_p['downgrade_needed']

threshold = meta['selected_threshold']

# Pipeline de inferencia
X_p_scaled = prep_p.transform(X_p)
probs_p = mod_p.predict_proba(X_p_scaled)[:, 1]
preds_p = (probs_p >= threshold).astype(int)

print("--- INFERENCIA PREDICTIVA ---")
for i, (idx, row) in enumerate(X_p.iterrows()):
    real_label = "downgrade_needed (1)" if y_true_p.iloc[i] == 1 else "maintain (0)"
    pred_label = "downgrade_needed (1)" if preds_p[i] == 1 else "maintain (0)"
    
    print(f"Muestra {i+1} | Input P10: {row['throughput_p10']:.2f} Mbps, Tendencia: {row['throughput_slope']:.4f}, Perfil Actual: {row['current_profile']}")
    print(f"           | Probabilidad de Degradación: {probs_p[i]:.4f} (Umbral: {threshold})")
    print(f"           | Predicción: {pred_label} (Real: {real_label})\n")


## 5. Conclusión
El comportamiento de inferencia es consistente y determinista, demostrando que los modelos están listos para ser utilizados por el agente de toma de decisiones en la **Fase 2**.